# Single-Neuron Regression Animation

Trains a one-input, one-output neuron (`nn.Linear(1, 1)`) on the historical delivery data and animates the fitted line converging over epochs.

Used in `modules/deep-learning/01-slides/02_neural_networks.qmd`.

Output: `modules/deep-learning/assets/nn_regression.gif`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib.animation import FuncAnimation

distances_list = [2.0, 3.0, 4.0, 5.0, 6.0]
times_list = [11.9, 15.3, 18.8, 22.2, 25.6]

distances = torch.tensor([[d] for d in distances_list], dtype=torch.float32)
times = torch.tensor([[t] for t in times_list], dtype=torch.float32)

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(1, 1))
# Start clearly wrong so the learning motion is visible
with torch.no_grad():
    model[0].weight.fill_(1.0)
    model[0].bias.fill_(8.0)

loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

x_line = torch.linspace(1.5, 7.0, 50).unsqueeze(1)
history = []
n_epochs = 100


def snapshot(epoch, loss_val):
    with torch.no_grad():
        y_line = model(x_line).squeeze().tolist()
        w = model[0].weight.item()
        b = model[0].bias.item()
    history.append(
        {"epoch": epoch, "y_line": y_line, "w": w, "b": b, "loss": loss_val}
    )


with torch.no_grad():
    start_y = model(x_line).squeeze().tolist()
    snapshot(0, loss_fn(model(distances), times).item())

for epoch in range(1, n_epochs + 1):
    optimizer.zero_grad()
    outputs = model(distances)
    loss = loss_fn(outputs, times)
    loss.backward()
    optimizer.step()
    snapshot(epoch, loss.item())

print(f"Frames: {len(history)}")
print(
    f"Start w={history[0]['w']:.3f}, b={history[0]['b']:.3f}, loss={history[0]['loss']:.2f}"
)
print(
    f"End   w={history[-1]['w']:.3f}, b={history[-1]['b']:.3f}, loss={history[-1]['loss']:.4f}"
)

In [ ]:
output_path = Path("../modules/deep-learning/assets/nn_regression.gif")
x_vals = x_line.squeeze().tolist()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    distances_list,
    times_list,
    s=120,
    color="#1f77b4",
    edgecolors="white",
    linewidth=1.5,
    zorder=3,
    label="Historical data",
)
ax.plot(
    x_vals,
    start_y,
    color="#ff7f0e",
    lw=2.0,
    linestyle=":",
    alpha=0.85,
    label="Starting prediction",
    zorder=2,
)
line, = ax.plot([], [], color="#ff7f0e", lw=2.5, label="Neuron prediction")
info = ax.text(
    0.02,
    0.98,
    "",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=11,
    family="monospace",
    bbox={
        "boxstyle": "round,pad=0.35",
        "facecolor": "white",
        "edgecolor": "#cccccc",
        "alpha": 0.92,
    },
)

ax.set_title("Single Neuron Learning Delivery Times")
ax.set_xlabel("Distance (miles)")
ax.set_ylabel("Time (minutes)")
ax.set_xlim(1.5, 7.0)
ax.set_ylim(5, 32)
ax.set_xticks([2.0, 3.0, 4.0, 5.0, 6.0, 7.0])
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()


def init():
    line.set_data([], [])
    info.set_text("")
    return line, info


def update(frame):
    snap = history[frame]
    line.set_data(x_vals, snap["y_line"])
    info.set_text(
        f"epoch {snap['epoch']:3d}\n"
        f"w = {snap['w']:6.3f}\n"
        f"b = {snap['b']:6.3f}\n"
        f"MSE = {snap['loss']:7.3f}"
    )
    return line, info


anim = FuncAnimation(
    fig,
    update,
    frames=len(history),
    init_func=init,
    interval=70,
    blit=True,
)

anim.save(output_path, writer="pillow", fps=12, dpi=120)
plt.close(fig)
print(f"Saved {output_path.resolve()} ({output_path.stat().st_size / 1024:.0f} KB)")